In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- CONFIGURATION ---
DATA_DIR = Path(".")
EXPERIMENTS = [
    {
        "label": r"Heavy-Tailed ($\alpha=1.2$)",
        "tasks_file": "gpm_alexnet_a1.2_s42_tasks.csv",
        "curves_file": "gpm_alexnet_a1.2_s42_curves.csv",
        "fallback_single": "gpm_alexnet_a1.2_s42.csv",
        "color": "#1f77b4",
    },
    {
        "label": r"Gaussian Baseline ($\alpha=2.0$)",
        "tasks_file": "gpm_alexnet_a2.0_s42_tasks.csv",
        "curves_file": "gpm_alexnet_a2.0_s42_curves.csv",
        "fallback_single": "gpm_alexnet_a2.0_s42.csv",
        "color": "#ff7f0e",
    },
]


def load_experiment_data(exp_cfg):
    tasks_path = DATA_DIR / exp_cfg["tasks_file"]
    curves_path = DATA_DIR / exp_cfg["curves_file"]
    single_path = DATA_DIR / exp_cfg["fallback_single"]

    if tasks_path.exists():
        df_tasks = pd.read_csv(tasks_path)
    elif single_path.exists():
        df_single = pd.read_csv(single_path)
        df_tasks = df_single.dropna(subset=["total_basis_rank"]).reset_index(
            drop=True
        )
    else:
        raise FileNotFoundError(f"Could not find milestone data for {exp_cfg['label']}")

    df_curves = None
    if curves_path.exists():
        df_curves = pd.read_csv(curves_path)
    elif single_path.exists():
        df_single = pd.read_csv(single_path)
        if "intra_task_acc" in df_single.columns:
            df_curves = df_single.dropna(subset=["intra_task_acc"]).reset_index(
                drop=True
            )

    return df_tasks, df_curves


def compute_seen_mean_acc(df_tasks):
    num_tasks = len(df_tasks)
    mean_accs = []
    for t_idx in range(num_tasks):
        acc_cols = [f"task_{j}_acc" for j in range(t_idx + 1)]
        row_mean = df_tasks.loc[t_idx, acc_cols].mean() * 100.0
        mean_accs.append(row_mean)
    return np.array(mean_accs)


def compute_task_auc(df_curves, num_tasks=20):
    if df_curves is None or "intra_task_acc" not in df_curves.columns:
        return None
    auc_per_task = []
    for t_idx in range(num_tasks):
        task_data = df_curves[df_curves["task_idx"] == t_idx].sort_values(
            by="epoch"
        )
        if len(task_data) > 0:
            epochs = task_data["epoch"].values
            accs = task_data["intra_task_acc"].values
            auc = np.trapezoid(accs, epochs) / (epochs[-1] - epochs[0])
            auc_per_task.append(auc * 100.0)
        else:
            auc_per_task.append(np.nan)
    return np.array(auc_per_task)


def compute_bwt_fwt_trajectories(df_tasks, num_tasks=20):
    """Computes Backward Transfer (BWT) and Zero-Shot Forward Transfer (FWT) across stream."""
    acc_matrix = np.zeros((num_tasks, num_tasks))
    for i in range(num_tasks):
        for j in range(num_tasks):
            col = f"task_{j}_acc"
            if col in df_tasks.columns and pd.notna(df_tasks.loc[i, col]):
                acc_matrix[i, j] = df_tasks.loc[i, col]

    # 1. Backward Transfer (BWT) after each task t (for t >= 1)
    bwt_traj = [np.nan]  # No BWT after Task 1
    for t in range(1, num_tasks):
        forgetting = [acc_matrix[t, j] - acc_matrix[j, j] for j in range(t)]
        bwt_traj.append(np.mean(forgetting) * 100.0)

    # 2. Zero-Shot Forward Transfer (FWT) after each task t (for t < num_tasks - 1)
    fwt_traj = []
    for t in range(num_tasks - 1):
        zero_shot_future = [acc_matrix[t, j] for j in range(t + 1, num_tasks)]
        fwt_traj.append(np.mean(zero_shot_future) * 100.0)
    fwt_traj.append(np.nan)  # No future tasks after Task 20

    return np.array(bwt_traj), np.array(fwt_traj), acc_matrix


# --- PLOTTING 2x2 GRID ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=300)
axes = axes.flatten()

for exp in EXPERIMENTS:
    df_tasks, df_curves = load_experiment_data(exp)
    num_tasks = len(df_tasks)
    task_indices = np.arange(1, num_tasks + 1)

    # 1. Average Retained Accuracy (Seen Tasks)
    mean_accs = compute_seen_mean_acc(df_tasks)
    axes[0].plot(
        task_indices,
        mean_accs,
        marker="o",
        label=exp["label"],
        color=exp["color"],
        linewidth=2,
    )

    # 2. Intra-Task Learning AUC
    auc_values = compute_task_auc(df_curves, num_tasks=num_tasks)
    if auc_values is not None:
        axes[1].plot(
            task_indices,
            auc_values,
            marker="s",
            label=exp["label"],
            color=exp["color"],
            linewidth=2,
        )

    # 3. Cumulative GPM Basis Rank
    cumulative_rank = df_tasks["total_basis_rank"].values
    axes[2].plot(
        task_indices,
        cumulative_rank,
        marker="^",
        label=exp["label"],
        color=exp["color"],
        linewidth=2,
    )

    # 4. Backward Transfer (Solid) & Forward Transfer (Dashed)
    bwt, fwt, _ = compute_bwt_fwt_trajectories(df_tasks, num_tasks=num_tasks)
    axes[3].plot(
        task_indices,
        bwt,
        marker="v",
        linestyle="-",
        label=f"{exp['label']} (BWT)",
        color=exp["color"],
        linewidth=2,
    )
    axes[3].plot(
        task_indices,
        fwt,
        marker="x",
        linestyle="--",
        label=f"{exp['label']} (Zero-Shot FWT)",
        color=exp["color"],
        alpha=0.75,
        linewidth=1.5,
    )

# --- FORMATTING PLOTS ---
axes[0].set_title(
    "Average Retained Accuracy (Seen Tasks)", fontsize=12, fontweight="bold"
)
axes[0].set_xlabel("Completed Task Index")
axes[0].set_ylabel("Mean Accuracy (%)")
axes[0].set_xticks(range(1, 21, 2))
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend(loc="lower left", frameon=True)

axes[1].set_title(
    "Intra-Task Learning AUC (Convergence Speed)",
    fontsize=12,
    fontweight="bold",
)
axes[1].set_xlabel("Task Index")
axes[1].set_ylabel("Normalized AUC (%)")
axes[1].set_xticks(range(1, 21, 2))
axes[1].grid(True, linestyle="--", alpha=0.6)
axes[1].legend(loc="lower right", frameon=True)

axes[2].set_title(
    "Cumulative GPM Basis Rank (Subspace Growth)",
    fontsize=12,
    fontweight="bold",
)
axes[2].set_xlabel("Completed Task Index")
axes[2].set_ylabel("Total Stored Rank")
axes[2].set_xticks(range(1, 21, 2))
axes[2].grid(True, linestyle="--", alpha=0.6)
axes[2].legend(loc="upper left", frameon=True)

axes[3].set_title(
    "Transfer Dynamics (BWT & Zero-Shot FWT)", fontsize=12, fontweight="bold"
)
axes[3].set_xlabel("Completed Task Index")
axes[3].set_ylabel("Transfer Metric (%)")
axes[3].set_xticks(range(1, 21, 2))
axes[3].axhline(0, color="gray", linestyle=":", alpha=0.7)
axes[3].grid(True, linestyle="--", alpha=0.6)
axes[3].legend(loc="center left", fontsize=9, frameon=True)

plt.tight_layout()
output_plot_path = DATA_DIR / "gpm_comprehensive_transfer_dynamics.png"
plt.savefig(output_plot_path, dpi=300)
plt.show()
print(f"2x2 Diagnostic plot successfully saved to: {output_plot_path}")

In [ ]:
def plot_scree_comparison(
    gaussian_snapshot_path,
    ht_snapshot_path,
    layer_key,
    save_path=None,
    normalize=True,
    cumulative=True,
):
    """Loads a Gaussian and a Heavy-Tailed snapshot, extracts singular values

    for a specified layer, and plots comparative scree and cumulative energy curves.
    """
    # 1. Load snapshot checkpoints
    snap_gauss = torch.load(gaussian_snapshot_path, map_location="cpu")
    snap_ht = torch.load(ht_snapshot_path, map_location="cpu")

    # Clean layer key format to match snapshot dict convention
    clean_layer_key = str(layer_key).replace(".", "_")

    if clean_layer_key not in snap_gauss["svd"]:
        available = list(snap_gauss["svd"].keys())
        raise KeyError(
            f"Layer '{clean_layer_key}' not found in snapshot SVD data. Available: {available}"
        )

    # 2. Extract singular values
    s_gauss = snap_gauss["svd"][clean_layer_key]["S"].numpy()
    s_ht = snap_ht["svd"][clean_layer_key]["S"].numpy()

    # Metadata for labels
    meta_gauss = snap_gauss.get("metadata", {})
    meta_ht = snap_ht.get("metadata", {})

    alpha_gauss = meta_gauss.get("alpha", 2.0)
    alpha_ht = meta_ht.get("alpha", "< 2.0")
    task_num = meta_gauss.get("task_number", meta_gauss.get("task_idx", "?"))

    # Optional normalization by top singular value (sigma_i / sigma_1)
    if normalize:
        s_gauss_plot = s_gauss / s_gauss[0]
        s_ht_plot = s_ht / s_ht[0]
        y_label = r"Normalized Singular Value $\sigma_i / \sigma_1$"
    else:
        s_gauss_plot = s_gauss
        s_ht_plot = s_ht
        y_label = r"Singular Value $\sigma_i$"

    # 3. Plotting
    num_panels = 2 if cumulative else 1
    fig, axes = plt.subplots(1, num_panels, figsize=(6.5 * num_panels, 4.5), dpi=300)
    if not cumulative:
        axes = [axes]

    # Panel 1: Scree Plot (Log-Linear)
    ax1 = axes[0]
    rank_gauss = np.arange(1, len(s_gauss_plot) + 1)
    rank_ht = np.arange(1, len(s_ht_plot) + 1)

    ax1.plot(
        rank_gauss,
        s_gauss_plot,
        label=rf"Gaussian ($\alpha={alpha_gauss}$)",
        color="#1f77b4",
        lw=2,
    )
    ax1.plot(
        rank_ht,
        s_ht_plot,
        label=rf"Heavy-Tailed ($\alpha={alpha_ht}$)",
        color="#d62728",
        lw=2,
    )

    ax1.set_yscale("log")
    ax1.set_xlabel("Singular Value Index $i$", fontsize=11)
    ax1.set_ylabel(y_label, fontsize=11)
    ax1.set_title(f"Layer: {layer_key} (Task {task_num}) - Spectral Decay", fontsize=12)
    ax1.grid(True, which="both", ls="--", alpha=0.4)
    ax1.legend(frameon=True, fontsize=10)

    # Panel 2: Cumulative Energy Explained
    if cumulative:
        ax2 = axes[1]
        cum_energy_gauss = np.cumsum(s_gauss**2) / np.sum(s_gauss**2)
        cum_energy_ht = np.cumsum(s_ht**2) / np.sum(s_ht**2)

        ax2.plot(
            rank_gauss,
            cum_energy_gauss,
            label=rf"Gaussian ($\alpha={alpha_gauss}$)",
            color="#1f77b4",
            lw=2,
        )
        ax2.plot(
            rank_ht,
            cum_energy_ht,
            label=rf"Heavy-Tailed ($\alpha={alpha_ht}$)",
            color="#d62728",
            lw=2,
        )

        # Standard GPM energy thresholds
        for th in [0.95, 0.99]:
            ax2.axhline(
                th,
                color="black",
                ls=":",
                alpha=0.6,
                label=f"Threshold {int(th * 100)}%" if th == 0.95 else None,
            )

        ax2.set_xlabel("Basis Rank $k$", fontsize=11)
        ax2.set_ylabel(
            r"Cumulative Energy $\sum_{i=1}^k \sigma_i^2 / \|\mathbf{R}\|_F^2$",
            fontsize=11,
        )
        ax2.set_title(
            f"Layer: {layer_key} (Task {task_num}) - Basis Capture", fontsize=12
        )
        ax2.set_ylim(None, 1.02)
        ax2.grid(True, ls="--", alpha=0.4)
        ax2.legend(frameon=True, fontsize=10)

    plt.tight_layout()

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved scree plot to {save_path}")

    plt.show()


# Example usage:
plot_scree_comparison(
    gaussian_snapshot_path="checkpoints/snapshots/snapshot_A2.0_T20_s0.pt",
    ht_snapshot_path="checkpoints/snapshots/snapshot_A1.2_T20_s0.pt",
    layer_key="4",  # or "layer1", "fc1", etc.
    # save_path="plots/scree_layer0_task5.pdf"
)

In [ ]:
import re
from pathlib import Path
from typing import Any, Optional, Union

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib import ticker
from scipy.signal import savgol_filter


def resolve_layer_keys_by_index(svd_dict, basis_dict, layer_query):
    """Maps SVD and Basis keys strictly by layer ordinal index."""
    svd_keys = list(svd_dict.keys())
    basis_keys = list(basis_dict.keys())

    # Map by direct index if querying by integer or matching string
    if isinstance(layer_query, int):
        idx = layer_query
    elif "classifier" in str(layer_query).lower() or "head" in str(
        layer_query
    ).lower():
        idx = -1
    else:
        # Extract digits: if 'features_8_weight', layer index in 10-layer stack is 8//2 = 4
        digits = re.findall(r"\d+", str(layer_query))
        num = int(digits[0]) if digits else 0
        idx = (
            num // 2
            if "features" in str(layer_query) and num % 2 == 0
            else min(num, len(svd_keys) - 1)
        )

    return svd_keys[idx], basis_keys[idx]


def compute_mode_alignment_from_svd(
    U_t: Union[torch.Tensor, np.ndarray],
    S_t: Union[torch.Tensor, np.ndarray],
    M_prev: Optional[Union[torch.Tensor, np.ndarray]],
    max_modes: int = 50,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Computes mode projection overlap p_i and energy-weighted alignment E_i."""
    if isinstance(U_t, torch.Tensor):
        U_t = U_t.detach().cpu().numpy()
    if isinstance(S_t, torch.Tensor):
        S_t = S_t.detach().cpu().numpy()
    if isinstance(M_prev, torch.Tensor):
        M_prev = M_prev.detach().cpu().numpy()

    U_t = np.asarray(U_t, dtype=np.float64)
    S_t = np.asarray(S_t, dtype=np.float64)

    num_modes = min(len(S_t), U_t.shape[1], max_modes)
    U_sub = U_t[:, :num_modes]
    sigmas_sq = S_t[:num_modes] ** 2

    if M_prev is None or M_prev.size == 0 or (M_prev.ndim > 1 and M_prev.shape[1] == 0):
        p_i = np.zeros(num_modes, dtype=np.float64)
    else:
        M_prev = np.asarray(M_prev, dtype=np.float64)
        if M_prev.ndim == 1:
            M_prev = M_prev[:, np.newaxis]

        if M_prev.shape[0] != U_sub.shape[0]:
            raise ValueError(
                f"Dimension mismatch: M_prev has ambient dim {M_prev.shape[0]}, "
                f"but U_t has ambient dim {U_sub.shape[0]}."
            )

        proj_coords = M_prev.T @ U_sub  # Shape: (K_prev, num_modes)
        p_i = np.sum(proj_coords**2, axis=0)
        p_i = np.clip(p_i, 0.0, 1.0)

    E_i = sigmas_sq * p_i
    return p_i, sigmas_sq, E_i


def load_snapshots_for_regime(
    snapshot_dir: Union[str, Path],
    alpha: float,
    seed: Optional[int] = 0,
    total_tasks: int = 20,
    epoch: Optional[int] = None,
) -> dict[int, dict[str, Any]]:
    """Loads snapshot checkpoint files mapped by task index."""
    snapshot_dir = Path(snapshot_dir)
    if not snapshot_dir.is_dir():
        raise FileNotFoundError(f"Snapshot directory does not exist: {snapshot_dir}")

    snapshots_by_task = {}
    epoch_tag = f"_E{epoch}" if epoch is not None else ""
    seed_tag = f"_s{seed}" if seed is not None else "*_s*"

    for t_num in range(total_tasks + 2):
        pattern = f"snapshot_A{alpha}_T{t_num:02d}{epoch_tag}{seed_tag}.pt"
        matches = sorted(snapshot_dir.glob(pattern))

        if not matches:
            fallback = f"*A{alpha}*T{t_num:02d}*{epoch_tag}{seed_tag}*.pt"
            matches = sorted(snapshot_dir.glob(fallback))

        for file_path in matches:
            data = torch.load(file_path, map_location="cpu", weights_only=False)
            t_idx = data["metadata"]["task_idx"]
            snapshots_by_task[t_idx] = data

    return snapshots_by_task


def run_temporal_mode_alignment_analysis(
    snapshot_dir: Union[str, Path],
    alpha: float,
    seed: Optional[int] = 0,
    layer_name: Union[str, int] = "features_8_weight",
    total_tasks: int = 20,
    max_modes: int = 40,
) -> dict[str, Any]:
    """Extracts mode alignment metrics across sequential tasks."""
    snapshots = load_snapshots_for_regime(
        snapshot_dir=snapshot_dir,
        alpha=alpha,
        seed=seed,
        total_tasks=total_tasks,
    )

    available_tasks = sorted(snapshots.keys())
    eval_tasks = [t for t in available_tasks if t >= 1 and (t - 1) in snapshots]

    if not eval_tasks:
        raise ValueError(
            f"No consecutive task pairs found for alpha={alpha}, seed={seed} in {snapshot_dir}"
        )

    tasks_out, p_i_list, sigmas_sq_list, E_i_list = [], [], [], []
    last_svd_k, last_basis_k = None, None

    for t in eval_tasks:
        snap_current = snapshots[t]
        snap_prev = snapshots[t - 1]

        svd_k, basis_k = resolve_layer_keys_by_index(
            snap_current["svd"], snap_prev["current_basis"], layer_name
        )
        last_svd_k, last_basis_k = svd_k, basis_k

        U_t = snap_current["svd"][svd_k]["U"]
        S_t = snap_current["svd"][svd_k]["S"]
        M_prev = snap_prev["current_basis"][basis_k]

        p_i, sigmas_sq, E_i = compute_mode_alignment_from_svd(
            U_t=U_t, S_t=S_t, M_prev=M_prev, max_modes=max_modes
        )

        tasks_out.append(t)
        p_i_list.append(p_i)
        sigmas_sq_list.append(sigmas_sq)
        E_i_list.append(E_i)

    return {
        "alpha": alpha,
        "tasks": np.array(tasks_out, dtype=int),
        "p_i": np.array(p_i_list),
        "sigmas_sq": np.array(sigmas_sq_list),
        "E_i": np.array(E_i_list),
        "resolved_svd_key": last_svd_k,
        "resolved_basis_key": last_basis_k,
    }


def smooth_series(
    y: np.ndarray, window_length: int = 15, polyorder: int = 2
) -> np.ndarray:
    """Applies Savitzky-Golay smoothing with dynamic window sizing."""
    n = len(y)
    if n < 5:
        return y
    w = min(window_length, n if n % 2 != 0 else n - 1)
    if w <= polyorder:
        w = polyorder + 2 if (polyorder + 2) % 2 != 0 else polyorder + 3
    if w > n:
        return y
    return savgol_filter(y, window_length=w, polyorder=polyorder)


def plot_mode_alignment_publication(
    res_ht: dict[str, Any],
    res_gauss: dict[str, Any],
    task_idx: int = 1,
    max_modes: int = 150,
    smooth_window: int = 15,
    save_path: Optional[Union[str, Path]] = None,
) -> None:
    """Generates a 3-panel publication-grade breakdown:

    1. Geometric Overlap (p_i) with smoothing.
    2. Cumulative Energy Fraction (reveals how fast each regime accumulates total power).
    3. Mode-Wise Projected Power Contribution (w_i * p_i).
    """
    ht_matches = np.where(res_ht["tasks"] == task_idx)[0]
    gauss_matches = np.where(res_gauss["tasks"] == task_idx)[0]

    if len(ht_matches) == 0 or len(gauss_matches) == 0:
        raise ValueError(f"Task {task_idx} not found in results.")

    ht_t = ht_matches[0]
    g_t = gauss_matches[0]

    k = min(max_modes, res_ht["p_i"].shape[1], res_gauss["p_i"].shape[1])
    modes = np.arange(1, k + 1)

    # 1. Raw & Smoothed Subspace Overlaps
    ht_p_raw = res_ht["p_i"][ht_t, :k]
    g_p_raw = res_gauss["p_i"][g_t, :k]

    ht_p_smooth = smooth_series(ht_p_raw, window_length=smooth_window)
    g_p_smooth = smooth_series(g_p_raw, window_length=smooth_window)

    # 2. Normalized Variance & Cumulative Energy
    ht_sig_sq = res_ht["sigmas_sq"][ht_t, :k]
    g_sig_sq = res_gauss["sigmas_sq"][g_t, :k]

    ht_var_frac = ht_sig_sq / res_ht["sigmas_sq"][ht_t].sum()
    g_var_frac = g_sig_sq / res_gauss["sigmas_sq"][g_t].sum()

    ht_cum_energy = np.cumsum(ht_var_frac)
    g_cum_energy = np.cumsum(g_var_frac)

    # 3. Mode-Wise Projected Power Contribution
    ht_proj_power = ht_var_frac * ht_p_raw
    g_proj_power = g_var_frac * g_p_raw

    # Setup Canvas: 3 Panels
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4.5), dpi=300)

    c_ht = "#1f77b4"
    c_gauss = "#d62728"

    # =========================================================================
    # Panel 1: Geometric Subspace Overlap (p_i)
    # =========================================================================
    # Raw faint traces
    ax1.plot(modes, ht_p_raw, color=c_ht, alpha=0.22, lw=1.0)
    ax1.plot(modes, g_p_raw, color=c_gauss, alpha=0.22, lw=1.0)

    # Bold smoothed trends
    ax1.plot(
        modes,
        ht_p_smooth,
        color=c_ht,
        lw=2.5,
        label=rf"Heavy-Tailed ($\alpha={res_ht.get('alpha', 1.2)}$)",
    )
    ax1.plot(
        modes,
        g_p_smooth,
        color=c_gauss,
        lw=2.5,
        label=rf"Gaussian ($\alpha={res_gauss.get('alpha', 2.0)}$)",
    )

    ax1.set_title(
        f"Geometric Mode Overlap ($p_i$)\nTask {task_idx}",
        fontsize=11,
        fontweight="bold",
    )
    ax1.set_xlabel("Singular Mode Index ($i$)", fontsize=10)
    ax1.set_ylabel(r"$p_i = \|\mathbf{M}_{t-1}^T \mathbf{u}_i\|^2$", fontsize=10)
    ax1.set_ylim(-0.02, 1.02)
    ax1.grid(True, linestyle="--", alpha=0.4)
    ax1.legend(frameon=True, fontsize=9, loc="upper right")

    # =========================================================================
    # Panel 2: Cumulative Energy / Power Fraction (CDF)
    # =========================================================================
    ax2.plot(
        modes,
        ht_cum_energy * 100,
        color=c_ht,
        lw=2.5,
        label=r"Heavy-Tailed ($\alpha=1.2$)",
    )
    ax2.plot(
        modes,
        g_cum_energy * 100,
        color=c_gauss,
        lw=2.5,
        label=r"Gaussian ($\alpha=2.0$)",
    )

    # Reference GPM threshold lines
    for thresh, style in zip(
        [80, 90, 95], [":", "--", "-."], strict=False
    ):
        ax2.axhline(
            thresh,
            color="gray",
            linestyle=style,
            alpha=0.6,
            lw=1.0,
            label=f"{thresh}% Threshold" if thresh == 95 else "",
        )

    # Shade the cumulative power gap
    ax2.fill_between(
        modes,
        g_cum_energy * 100,
        ht_cum_energy * 100,
        where=ht_cum_energy >= g_cum_energy,
        color=c_ht,
        alpha=0.12,
        label="Power Condensation Gain",
    )

    ax2.set_title(
        f"Cumulative Energy Fraction\nTask {task_idx}",
        fontsize=11,
        fontweight="bold",
    )
    ax2.set_xlabel("Singular Mode Index ($i$)", fontsize=10)
    ax2.set_ylabel(r"Cumulative Variance ($\sum_{j i} \sigma_j^2 / \sum \sigma^2$ %)", fontsize=10)
    ax2.set_ylim(0, 103)
    ax2.yaxis.set_major_formatter(ticker.PercentFormatter())
    ax2.grid(True, linestyle="--", alpha=0.4)
    ax2.legend(frameon=True, fontsize=8.5, loc="lower right")

    # =========================================================================
    # Panel 3: Projected Power Contribution (w_i * p_i)
    # =========================================================================
    ax3.plot(
        modes,
        ht_proj_power * 100,
        color=c_ht,
        lw=2.2,
        marker="s",
        markersize=3.5,
        label=r"Heavy-Tailed ($\alpha=1.2$)",
    )
    ax3.plot(
        modes,
        g_proj_power * 100,
        color=c_gauss,
        lw=2.2,
        marker="o",
        markersize=3.5,
        label=r"Gaussian ($\alpha=2.0$)",
    )

    ax3.set_title(
        f"Projected Mode Power ($w_i \cdot p_i$)\nTask {task_idx}",
        fontsize=11,
        fontweight="bold",
    )
    ax3.set_xlabel("Singular Mode Index ($i$)", fontsize=10)
    ax3.set_ylabel("Variance Contribution to Projection (%)", fontsize=10)
    ax3.set_yscale("log")
    ax3.grid(True, linestyle="--", alpha=0.4)
    ax3.legend(frameon=True, fontsize=9, loc="upper right")

    plt.tight_layout()

    if save_path:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved figure to {save_path}")

    plt.show()


if __name__ == "__main__":
    snapshot_dir = "checkpoints/mlp"
    if Path(snapshot_dir).exists():
        res_ht = run_temporal_mode_alignment_analysis(
            snapshot_dir=snapshot_dir,
            alpha=1.2,
            seed=0,
            layer_name="4",
            max_modes=150,
        )
        res_gauss = run_temporal_mode_alignment_analysis(
            snapshot_dir=snapshot_dir,
            alpha=2.0,
            seed=0,
            layer_name="4",
            max_modes=150,
        )
        plot_mode_alignment_publication(
            res_ht=res_ht,
            res_gauss=res_gauss,
            task_idx=1,
            max_modes=150,
            save_path="mode_alignment_task1.png",
        )

In [ ]:
def plot_layer_dynamics_across_tasks(
    df_metrics,
    layer_name,
    save_path=None,
    alpha_gauss=2.0,
    alpha_ht=1.2,
):
    """Plots the trajectory of Stable Rank, Effective Rank, and Spectral Entropy

    across sequential tasks (0 to T) for a specified reference layer.
    """
    # 1. Filter DataFrame for the target layer
    clean_layer_name = str(layer_name).replace(".", "_")
    sub_df = df_metrics[df_metrics["layer"] == clean_layer_name].copy()

    if sub_df.empty:
        available = df_metrics["layer"].unique()
        raise ValueError(
            f"Layer '{clean_layer_name}' not found. Available layers: {available}"
        )

    # 2. Separate Gaussian and Heavy-Tailed subsets
    df_g = sub_df[sub_df["alpha"] == float(alpha_gauss)]
    df_ht = sub_df[sub_df["alpha"] == float(alpha_ht)]

    metrics = [
        ("srank", r"Stable Rank $\operatorname{srank}(\mathbf{R})$"),
        ("erank", r"Effective Rank $\operatorname{erank}(\mathbf{R})$"),
        ("entropy", r"Spectral Entropy $S(\mathbf{R})$"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), dpi=300)

    for ax, (col_name, y_label) in zip(axes, metrics):
        # Aggregate across seeds (mean and standard error)
        g_stats = (
            df_g.groupby("task")[col_name].agg(["mean", "sem", "count"]).reset_index()
        )
        ht_stats = (
            df_ht.groupby("task")[col_name].agg(["mean", "sem", "count"]).reset_index()
        )

        # Plot Gaussian trajectory
        ax.plot(
            g_stats["task"],
            g_stats["mean"],
            marker="o",
            markersize=4,
            lw=2,
            color="#1f77b4",
            label=rf"Gaussian ($\alpha={alpha_gauss}$)",
        )
        if g_stats["sem"].notna().any() and (g_stats["count"] > 1).any():
            ax.fill_between(
                g_stats["task"],
                g_stats["mean"] - g_stats["sem"],
                g_stats["mean"] + g_stats["sem"],
                color="#1f77b4",
                alpha=0.18,
            )

        # Plot Heavy-Tailed trajectory
        ax.plot(
            ht_stats["task"],
            ht_stats["mean"],
            marker="s",
            markersize=4,
            lw=2,
            color="#d62728",
            label=rf"Heavy-Tailed ($\alpha={alpha_ht}$)",
        )
        if ht_stats["sem"].notna().any() and (ht_stats["count"] > 1).any():
            ax.fill_between(
                ht_stats["task"],
                ht_stats["mean"] - ht_stats["sem"],
                ht_stats["mean"] + ht_stats["sem"],
                color="#d62728",
                alpha=0.18,
            )

        ax.set_xlabel("Task Horizon ($t$)", fontsize=11)
        ax.set_ylabel(y_label, fontsize=11)
        ax.grid(True, ls="--", alpha=0.4)
        ax.legend(frameon=True, fontsize=10)

        # Format x-ticks to display integer task indices cleanly
        all_tasks = sorted(sub_df["task"].unique())
        if len(all_tasks) > 0:
            ax.set_xticks(np.arange(min(all_tasks), max(all_tasks) + 1, 2))

    fig.suptitle(
        f"Representation Variance Dynamics across Tasks — Layer: {layer_name}",
        fontsize=13,
        y=1.02,
    )
    plt.tight_layout()

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved figure to {save_path}")

    plt.show()


# Example Usage:
df_metrics = pd.read_csv("spectral_metrics_summary.csv")
plot_layer_dynamics_across_tasks(
    df_metrics=df_metrics,
    layer_name="4",  # Middle layer key in your MLP
    # save_path="plots/dynamics_layer2_task0_to_20.pdf",
    alpha_gauss=2.0,
    alpha_ht=1.2,
)

In [ ]:
from pathlib import Path
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def parse_layer_depth(layer_name):
    """Assigns an integer depth index for sorting layers sequentially,

    ensuring the classifier/head layer is placed at the final depth.
    """
    name_str = str(layer_name).lower()

    if "classifier" in name_str or "head" in name_str or "fc_last" in name_str:
        return 9999

    digits = re.findall(r"\d+", name_str)
    if digits:
        return int(digits[0])

    return 5000


def plot_spectral_metrics_across_depth(
    df_metrics,
    task_idx=None,
    save_path=None,
    alpha_gauss=2.0,
    alpha_ht=1.2,
):
    """Plots Stable Rank, Effective Rank, and Spectral Entropy across network depth

    for a specified task (defaults to the final task in df_metrics).
    """
    # 1. Determine target task
    if task_idx is None:
        target_task = int(df_metrics["task"].max())
    else:
        target_task = int(task_idx)

    sub_df = df_metrics[df_metrics["task"] == target_task].copy()
    if sub_df.empty:
        raise ValueError(
            f"No data found for Task {target_task}. Available tasks: {sorted(df_metrics['task'].unique())}"
        )

    # 2. Sort layers by architectural depth
    sub_df["depth_order"] = sub_df["layer"].apply(parse_layer_depth)
    sub_df.sort_values(by="depth_order", inplace=True)

    ordered_layers = sub_df["layer"].unique().tolist()
    layer_display_labels = [
        "Classifier" if parse_layer_depth(l) == 9999 else f"Layer {l}"
        for l in ordered_layers
    ]
    depth_x = np.arange(len(ordered_layers))

    # 3. Filter for initialisation regimes
    df_g = sub_df[sub_df["alpha"] == float(alpha_gauss)]
    df_ht = sub_df[sub_df["alpha"] == float(alpha_ht)]

    metrics = [
        ("srank", r"Stable Rank $\operatorname{srank}(\mathbf{R})$"),
        ("erank", r"Effective Rank $\operatorname{erank}(\mathbf{R})$"),
        ("entropy", r"Spectral Entropy $S(\mathbf{R})$"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), dpi=300)

    for ax, (col_name, y_label) in zip(axes, metrics):
        # Aggregate across seeds
        g_stats = (
            df_g.groupby("layer")[col_name]
            .agg(["mean", "sem", "count"])
            .reindex(ordered_layers)
            .reset_index()
        )
        ht_stats = (
            df_ht.groupby("layer")[col_name]
            .agg(["mean", "sem", "count"])
            .reindex(ordered_layers)
            .reset_index()
        )

        # Plot Gaussian across depth
        ax.plot(
            depth_x,
            g_stats["mean"],
            marker="o",
            markersize=5,
            lw=2,
            color="#1f77b4",
            label=rf"Gaussian ($\alpha={alpha_gauss}$)",
        )
        if g_stats["sem"].notna().any() and (g_stats["count"] > 1).any():
            ax.fill_between(
                depth_x,
                g_stats["mean"] - g_stats["sem"],
                g_stats["mean"] + g_stats["sem"],
                color="#1f77b4",
                alpha=0.18,
            )

        # Plot Heavy-Tailed across depth
        ax.plot(
            depth_x,
            ht_stats["mean"],
            marker="s",
            markersize=5,
            lw=2,
            color="#d62728",
            label=rf"Heavy-Tailed ($\alpha={alpha_ht}$)",
        )
        if ht_stats["sem"].notna().any() and (ht_stats["count"] > 1).any():
            ax.fill_between(
                depth_x,
                ht_stats["mean"] - ht_stats["sem"],
                ht_stats["mean"] + ht_stats["sem"],
                color="#d62728",
                alpha=0.18,
            )

        ax.set_xlabel("Network Depth", fontsize=11)
        ax.set_ylabel(y_label, fontsize=11)
        ax.set_xticks(depth_x)
        ax.set_xticklabels(layer_display_labels, fontsize=10)
        ax.grid(True, ls="--", alpha=0.4)
        ax.legend(frameon=True, fontsize=10)

    fig.suptitle(
        f"Representation Variance Profiles Across Network Depth (Task {target_task})",
        fontsize=13,
        y=1.02,
    )
    plt.tight_layout()

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved depth profile figure to {save_path}")

    plt.show()


# Example Execution:
df_metrics = pd.read_csv("spectral_metrics_summary.csv")
plot_spectral_metrics_across_depth(
    df_metrics=df_metrics,
    task_idx=20,  # Or None to auto-select the final task
    # save_path="plots/spectral_depth_task20.pdf",
    alpha_gauss=2.0,
    alpha_ht=1.2,
)

In [ ]:
import matplotlib.colors as mcolors
from scipy.interpolate import griddata


def plot_capacity_difference_heatmap(
    df_metrics,
    task_idx=1,
    min_energy=0.80,
    max_energy=0.999,
    alpha_gauss=2.0,
    alpha_ht=1.2,
    save_path=None,
    grid_resolution=(150, 150),
):
    """Generates a 2D Heatmap of Basis Savings Delta k = k_Gauss - k_HT

    across Network Depth (x-axis) vs Inverted Logarithmic Energy 1 - epsilon
    (y-axis).
    """
    # 1. Filter for the target task
    sub_df = df_metrics[df_metrics["task"] == int(task_idx)].copy()
    if sub_df.empty:
        raise ValueError(f"No records found for Task {task_idx}")

    # 2. Sort layers sequentially
    sub_df["depth_order"] = sub_df["layer"].apply(parse_layer_depth)
    sub_df.sort_values(by="depth_order", inplace=True)

    ordered_layers = sub_df["layer"].unique().tolist()
    layer_labels = [
        "Head" if parse_layer_depth(l) == 9999 else f"L{l}" for l in ordered_layers
    ]
    depth_indices = np.arange(len(ordered_layers))
    layer_to_idx = {l: i for i, l in enumerate(ordered_layers)}

    # 3. Detect energy threshold columns within [min_energy, max_energy]
    th_cols = [c for c in df_metrics.columns if c.startswith("k_")]
    valid_thresholds = []
    for c in th_cols:
        eps = parse_column_energy(c)
        if eps is not None and (min_energy <= eps <= max_energy):
            valid_thresholds.append((eps, c))

    if not valid_thresholds:
        raise ValueError(
            f"No threshold columns found in range [{min_energy}, {max_energy}]"
        )

    valid_thresholds.sort(key=lambda x: x[0])

    df_g = sub_df[sub_df["alpha"] == float(alpha_gauss)]
    df_ht = sub_df[sub_df["alpha"] == float(alpha_ht)]

    # 4. Construct point grid: Depth Index x Inverted Log Remaining Energy log10(1 - eps)
    points_x, points_y, delta_k_vals = [], [], []

    for eps, col_key in valid_thresholds:
        g_means = df_g.groupby("layer")[col_key].mean().reindex(ordered_layers)
        ht_means = df_ht.groupby("layer")[col_key].mean().reindex(ordered_layers)
        delta_k = (g_means - ht_means).to_numpy()

        y_val = 1.0 - eps  # Remaining energy fraction
        for d_idx, d_k in zip(depth_indices, delta_k):
            points_x.append(d_idx)
            points_y.append(y_val)
            delta_k_vals.append(d_k)

    points_x = np.array(points_x)
    points_y = np.array(points_y)
    delta_k_vals = np.array(delta_k_vals)

    # 5. Continuous 2D grid interpolation
    grid_x, grid_y = np.meshgrid(
        np.linspace(depth_indices.min(), depth_indices.max(), grid_resolution[0]),
        np.geomspace(points_y.min(), points_y.max(), grid_resolution[1]),
    )

    # Interpolate in log-space for smooth vertical transitions
    grid_z = griddata(
        (points_x, np.log10(points_y)),
        delta_k_vals,
        (grid_x, np.log10(grid_y)),
        method="cubic",
    )

    # 6. Plotting
    fig, ax = plt.subplots(figsize=(8, 7), dpi=300)
    ax.set_box_aspect(1)

    # Symmetrical colormap range centered at Delta k = 0
    max_abs_delta = np.nanmax(np.abs(delta_k_vals))
    norm = mcolors.TwoSlopeNorm(vcenter=0.0, vmin=-max_abs_delta, vmax=max_abs_delta)

    mesh = ax.pcolormesh(
        grid_x,
        grid_y,
        grid_z,
        shading="gouraud",
        cmap="seismic",  # Blue = Heavy-Tailed Savings (>0), Red = Gaussian Lower (<0)
        norm=norm,
    )

    # Contour lines showing boundary of zero difference
    ax.contour(
        grid_x,
        grid_y,
        grid_z,
        levels=[0.0],
        colors="black",
        linewidths=1.2,
        linestyles="--",
    )

    # Configure Inverted Logarithmic Y-Axis
    ax.set_yscale("log")
    ax.invert_yaxis()  # Top of plot = high energy (99.9%), Bottom = lower energy (80%)

    # Custom tick positions for intuitive percentage readings
    target_ticks_pct = [0.80, 0.90, 0.95, 0.98, 0.99, 0.995, 0.999]
    actual_ticks = [
        1.0 - p
        for p in target_ticks_pct
        if points_y.min() <= (1.0 - p) <= points_y.max()
    ]
    ax.yaxis.set_major_locator(ticker.FixedLocator(actual_ticks))
    ax.yaxis.set_major_formatter(
        ticker.FuncFormatter(lambda y, _: f"{((1.0 - y) * 100):.1f}%")
    )
    ax.yaxis.set_minor_locator(ticker.NullLocator())

    ax.set_xticks(depth_indices)
    ax.set_xticklabels(layer_labels, fontsize=10.5)

    ax.set_xlabel("Network Depth", fontsize=11, labelpad=8)
    ax.set_ylabel(
        r"Energy Threshold ($\epsilon$, expanded log tail)",
        fontsize=11,
        labelpad=8,
    )
    ax.set_title(
        f"GPM Capacity Delta Map: $\Delta k_\epsilon = k_{{\mathrm{{Gauss}}}} - k_{{\mathrm{{HT}}}}$ (Task {task_idx})",
        fontsize=12,
        fontweight="semibold",
        pad=12,
    )

    cbar = fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(
        r"Basis Vectors Saved by Heavy Tails ($\Delta k_\epsilon$)",
        fontsize=10.5,
        labelpad=8,
    )

    plt.tight_layout()

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved capacity delta heatmap to {save_path}")

    plt.show()


# Example Execution:
df_metrics = pd.read_csv("spectral_metrics_summary.csv")
plot_capacity_difference_heatmap(
    df_metrics=df_metrics,
    task_idx=1,
    min_energy=0.80,
    max_energy=0.999,
    # save_path="plots/capacity_delta_heatmap_task1.pdf"
)

In [ ]:
def get_ordered_layer_columns(df):
    """Detects and sorts total/projected variance column pairs by layer depth."""
    total_cols = [c for c in df.columns if c.startswith("var_total_")]
    layer_keys = [c.replace("var_total_", "") for c in total_cols]
    layer_keys.sort(key=parse_layer_depth)

    display_labels = [
        "Head" if parse_layer_depth(k) == 9999 else f"L{parse_layer_depth(k) // 2}"
        for k in layer_keys
    ]
    return layer_keys, display_labels


def plot_representation_projection_energy(
    csv_ht_path="gpm_a1.2_run_s0.csv",
    csv_gauss_path="gpm_a2.0_run_s0.csv",
    depth_task_idx=1,
    mid_layer_key="features_8_weight",
    save_path=None,
):
    """Plots Representation Projection Energy Ratio (E_proj = var_proj / var_total)

    across both the temporal task axis and the depth axis.
    """
    # 1. Load and clean task completion rows
    df_ht = pd.read_csv(csv_ht_path).dropna(subset=["task_idx"]).sort_values("task_idx")
    df_g = (
        pd.read_csv(csv_gauss_path).dropna(subset=["task_idx"]).sort_values("task_idx")
    )

    layer_keys, layer_display_labels = get_ordered_layer_columns(df_ht)

    # 2. Extract Temporal Dynamics (Exclude task 0 where basis is uninitialized)
    df_ht_temp = df_ht[df_ht["task_idx"] >= 1].copy()
    df_g_temp = df_g[df_g["task_idx"] >= 1].copy()

    tasks_temp = df_ht_temp["task_idx"].astype(int).to_numpy()

    ht_temp_proj = (
        df_ht_temp[f"var_proj_{mid_layer_key}"]
        / df_ht_temp[f"var_total_{mid_layer_key}"]
    ).to_numpy()
    g_temp_proj = (
        df_g_temp[f"var_proj_{mid_layer_key}"] / df_g_temp[f"var_total_{mid_layer_key}"]
    ).to_numpy()

    # 3. Extract Depth Profile for Selected Task
    row_ht_task = df_ht[df_ht["task_idx"] == depth_task_idx]
    row_g_task = df_g[df_g["task_idx"] == depth_task_idx]

    if row_ht_task.empty or row_g_task.empty:
        raise ValueError(f"Task {depth_task_idx} not found in one or both CSV files.")

    ht_depth_proj = np.array(
        [
            (row_ht_task[f"var_proj_{k}"] / row_ht_task[f"var_total_{k}"]).iloc[0]
            for k in layer_keys
        ]
    )
    g_depth_proj = np.array(
        [
            (row_g_task[f"var_proj_{k}"] / row_g_task[f"var_total_{k}"]).iloc[0]
            for k in layer_keys
        ]
    )

    depth_x = np.arange(len(layer_keys))
    x_dense = np.linspace(depth_x.min(), depth_x.max(), 300)

    # Smooth splines for depth profile
    spline_ht = make_interp_spline(depth_x, ht_depth_proj, k=3)
    spline_g = make_interp_spline(depth_x, g_depth_proj, k=3)

    # 4. Canvas Setup: Side-by-Side Panels
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), dpi=300)

    color_gauss = "#d62728"
    color_ht = "#1f77b4"

    # --- Panel A: Temporal Evolution ---
    ax1.plot(
        tasks_temp,
        g_temp_proj,
        marker="o",
        markersize=5,
        color=color_gauss,
        lw=2.2,
        label=r"Gaussian ($\alpha=2.0$)",
    )
    ax1.plot(
        tasks_temp,
        ht_temp_proj,
        marker="s",
        markersize=5,
        color=color_ht,
        lw=2.2,
        label=r"Heavy-Tailed ($\alpha=1.2$)",
    )

    ax1.fill_between(
        tasks_temp,
        g_temp_proj,
        ht_temp_proj,
        where=ht_temp_proj >= g_temp_proj,
        color=color_ht,
        alpha=0.14,
        label=r"Subspace Alignment Gain ($\Delta E_{\mathrm{proj}}$)",
    )

    ax1.set_title(
        f"Temporal Subspace Recycling ({layer_display_labels[layer_keys.index(mid_layer_key)]})",
        fontsize=12,
        fontweight="semibold",
        pad=10,
    )
    ax1.set_xlabel("Task Index ($t$)", fontsize=11, labelpad=8)
    ax1.set_ylabel(
        r"Projection Energy Ratio $E_{\mathrm{proj}} = \frac{\|\mathbf{M}_{t-1}^T \mathbf{R}_t\|_F^2}{\|\mathbf{R}_t\|_F^2}$",
        fontsize=11,
        labelpad=8,
    )
    ax1.set_xticks(tasks_temp[::2])
    ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"{y * 100:.0f}%"))
    ax1.grid(True, linestyle="--", alpha=0.35)
    ax1.legend(frameon=True, fontsize=9.5, loc="lower right")

    # --- Panel B: Depth Profile ---
    ax2.plot(
        x_dense,
        np.clip(spline_g(x_dense), 0, 1),
        color=color_gauss,
        lw=2.2,
        label=r"Gaussian ($\alpha=2.0$)",
    )
    ax2.scatter(depth_x, g_depth_proj, color=color_gauss, s=32, zorder=5)

    ax2.plot(
        x_dense,
        np.clip(spline_ht(x_dense), 0, 1),
        color=color_ht,
        lw=2.2,
        label=r"Heavy-Tailed ($\alpha=1.2$)",
    )
    ax2.scatter(depth_x, ht_depth_proj, color=color_ht, s=32, marker="s", zorder=5)

    ax2.fill_between(
        x_dense,
        np.clip(spline_g(x_dense), 0, 1),
        np.clip(spline_ht(x_dense), 0, 1),
        where=spline_ht(x_dense) >= spline_g(x_dense),
        color=color_ht,
        alpha=0.14,
        label=r"Subspace Alignment Gain ($\Delta E_{\mathrm{proj}}$)",
    )

    ax2.set_title(
        f"Depth-Wise Subspace Overlap (Task {depth_task_idx})",
        fontsize=12,
        fontweight="semibold",
        pad=10,
    )
    ax2.set_xlabel("Network Depth", fontsize=11, labelpad=8)
    ax2.set_ylabel(
        r"Projection Energy Ratio $E_{\mathrm{proj}}$", fontsize=11, labelpad=8
    )
    ax2.set_xticks(depth_x)
    ax2.set_xticklabels(layer_display_labels, fontsize=10)
    ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda y, _: f"{y * 100:.0f}%"))
    ax2.grid(True, linestyle="--", alpha=0.35)
    ax2.legend(frameon=True, fontsize=9.5, loc="lower right")

    plt.tight_layout()

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, bbox_inches="tight")
        print(f"Saved alignment figure to {save_path}")

    plt.show()


# Example Execution:
plot_representation_projection_energy(
    csv_ht_path="gpm_a1.2_run_s0.csv",
    csv_gauss_path="gpm_a2.0_run_s0.csv",
    depth_task_idx=1,
    mid_layer_key="features_8_weight",
    save_path="plots/representation_projection_energy.pdf",
)

In [ ]:
# --- 1. CONFIGURATION & PATHS ---
RESULTS_DIR = Path("./phase_sweep/results")

# Match the exact grid parameters used in your sweep
ALPHA_VALS = np.round(np.arange(1.0, 2.01, 0.1), 2)  # 1.0 to 2.0 (11 steps)
G_VALS = np.round(np.arange(0.5, 3.01, 0.25), 2)  # 0.5 to 3.0 (11 steps)

NUM_ALPHA = len(ALPHA_VALS)
NUM_G = len(G_VALS)

# Mapping indices for array insertion
alpha_to_idx = {a: i for i, a in enumerate(ALPHA_VALS)}
g_to_idx = {g: j for j, g in enumerate(G_VALS)}

# Matrices configured for FLIPPED AXES: Rows = g (vertical), Columns = alpha (horizontal)
matrix_final_acc = np.full((NUM_G, NUM_ALPHA), np.nan)
matrix_task20_auc = np.full((NUM_G, NUM_ALPHA), np.nan)
matrix_hidden_rank = np.full((NUM_G, NUM_ALPHA), np.nan)


# --- 2. DATA AGGREGATION MATCHING YOUR CSV HEADERS ---
if not RESULTS_DIR.exists():
    raise FileNotFoundError(f"Results directory not found at {RESULTS_DIR}")

csv_files = list(RESULTS_DIR.glob("results_a*_g*.csv"))
print(f"Found {len(csv_files)} result CSV files. Parsing matching headers...")

for csv_path in csv_files:
    # Parse alpha and g from filename (e.g., results_a1.20_g1.50.csv)
    filename = csv_path.stem
    parts = filename.split("_")
    alpha_val = float(parts[1].replace("a", ""))
    g_val = float(parts[2].replace("g", ""))

    if alpha_val not in alpha_to_idx or g_val not in g_to_idx:
        continue

    # Note the flipped index order: row = g, col = alpha
    row_g = g_to_idx[g_val]
    col_a = alpha_to_idx[alpha_val]

    df = pd.read_csv(csv_path)

    # Match exact accuracy columns: task_0_acc ... task_19_acc
    acc_cols = [f"task_{t}_acc" for t in range(20) if f"task_{t}_acc" in df.columns]

    # A. Final Average Accuracy (Mean across all 20 tasks in the absolute final state)
    if acc_cols:
        final_row = df.iloc[-1]
        matrix_final_acc[row_g, col_a] = final_row[acc_cols].mean()

    # B. Final Task (Task 19) Plasticity / Trajectory AUC
    # Measures the normalized Area Under the Curve (AUC) for Task 19 during its training phase
    if "task_19_acc" in df.columns:
        t20_values = df["task_19_acc"].dropna()
        if len(t20_values) > 0:
            # Take the final evaluated accuracy score for Task 20
            matrix_task20_auc[row_g, col_a] = t20_values.iloc[-1]

    # C. Hidden Layer Reserved Rank (Excludes Layer 1: basis_rank_features.0.weight)
    if (
        "total_basis_rank" in df.columns
        and "basis_rank_features.0.weight" in df.columns
    ):
        final_total_rank = df["total_basis_rank"].iloc[-1]
        final_layer1_rank = df["basis_rank_features.0.weight"].iloc[-1]
        matrix_hidden_rank[row_g, col_a] = final_total_rank - final_layer1_rank
    elif "total_basis_rank" in df.columns:
        matrix_hidden_rank[row_g, col_a] = df["total_basis_rank"].iloc[-1]

print("Grid aggregation complete!")


# --- 3. FLIPPED AXES PHASE DIAGRAM PLOTTING ROUTINE ---
def plot_phase_heatmap(matrix_data, title, cbar_label, cmap="viridis", fmt=".2f"):
    plt.figure(figsize=(10, 7.5))

    # Heatmap setup: Y-axis = g (vertical), X-axis = alpha (horizontal)
    ax = sns.heatmap(
        matrix_data,
        xticklabels=ALPHA_VALS,
        yticklabels=G_VALS,
        annot=True,
        fmt=fmt,
        cmap=cmap,
        linewidths=0.5,
        linecolor="white",
        cbar_kws={"label": cbar_label},
        annot_kws={"size": 8.5, "weight": "bold"},
    )

    # Invert Y-axis so g increases upwards (standard physics convention)
    ax.invert_yaxis()

    plt.xlabel(r"Tail Exponent ($\alpha$)", fontweight="bold", fontsize=12)
    plt.ylabel(r"Initialization Gain ($g$)", fontweight="bold", fontsize=12)
    plt.title(title, fontweight="bold", fontsize=13, pad=14)

    plt.tight_layout()
    plt.show()


# --- 4. GENERATE THE THREE PHASE DIAGRAMS ---

# 1. Final Average Accuracy Phase Diagram
plot_phase_heatmap(
    matrix_final_acc,
    title="Phase Diagram: Final 20-Task Average Test Accuracy\n"
    "Maps Continual Learning Performance Across Parameter Space",
    cbar_label="Mean Test Accuracy",
    cmap="magma",
    fmt=".3f",
)

# 2. Final Task (Task 20) Plasticity / Trajectory AUC Phase Diagram
plot_phase_heatmap(
    matrix_task20_auc,
    title="Phase Diagram: Task 20 Accuracy\n"
    "Identifies Capacity Exhaustion vs. Retained Learning Ability",
    cbar_label="Task 20 Final Accuracy",
    cmap="plasma",
    fmt=".3f",
)

# 3. Hidden Layer Reserved Basis Rank Phase Diagram (Excludes Layer 1)
plot_phase_heatmap(
    matrix_hidden_rank,
    title="Phase Diagram: Hidden Layer Reserved Basis Rank ($K_{\\text{hidden}}$)\n"
    "Quantifies Subspace Compression (Excludes Invariant Layer 1)",
    cbar_label="Hidden Layer Basis Vectors",
    cmap="viridis_r",  # Reversed so lower rank (higher compression) stands out
    fmt=".0f",
)

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- 1. CONFIGURATION ---
GAUSSIAN_CSV = "gpm_a2.0_run_s0.csv"  # Standard Gaussian initialization run
HEAVY_TAIL_CSV = "gpm_a1.2_run_s0.csv"  # Heavy-Tailed initialization run
NUM_TASKS = 20


# --- 2. EXTRACTOR FUNCTION FOR A SINGLE CSV FILE ---
def extract_run_metrics(filepath, num_tasks=20):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"Could not find target CSV file at: '{filepath}'")

    df = pd.read_csv(filepath)
    num_entries = len(df)

    # Sort chronological accuracy columns
    task_acc_cols = [
        c for c in df.columns if c.startswith("task_") and c.endswith("_acc")
    ]
    task_acc_cols = sorted(task_acc_cols, key=lambda x: int(x.split("_")[1]))

    # Determine active task footprint per row to find task completion boundaries
    active_tasks_per_step = np.array(
        [df.iloc[idx][task_acc_cols].notna().sum() for idx in range(num_entries)]
    )

    # Detect transition step indices where each task finishes training
    transition_indices = []
    current_num = active_tasks_per_step[0]
    for idx in range(1, num_entries):
        if active_tasks_per_step[idx] > current_num:
            transition_indices.append(idx - 1)
            current_num = active_tasks_per_step[idx]
    transition_indices.append(num_entries - 1)

    if len(transition_indices) != num_tasks:
        print(
            f"Warning: Boundary mismatch in {filepath}. Detected {len(transition_indices)} boundaries instead of {num_tasks}"
        )

    # Determine the step intervals for each task
    task_step_intervals = []
    start_idx = 0
    for end_idx in transition_indices:
        task_step_intervals.append((start_idx, end_idx))
        start_idx = end_idx + 1

    # Storage arrays
    avg_acc_at_wrapup = []
    current_task_auc = []
    total_basis_rank = []
    layer_1_basis_rank = []

    # Detect available rank columns dynamically
    layer_1_col = "basis_rank_0" if "basis_rank_0" in df.columns else None
    if layer_1_col is None:
        layer_1_cols = [
            c
            for c in df.columns
            if "basis" in c and ("0" in c or "layer_0" in c or "fc1" in c)
        ]
        layer_1_col = layer_1_cols[0] if len(layer_1_cols) > 0 else None

    for t_idx, boundary_idx in enumerate(transition_indices):
        # 1. Global Average Accuracy across all active tasks
        row_accs = df.iloc[boundary_idx][task_acc_cols].values
        active_accs = row_accs[~pd.isna(row_accs)]
        avg_acc_at_wrapup.append(np.mean(active_accs) if len(active_accs) > 0 else 0.0)

        # 2. Plasticity AUC during training of Task t_idx
        start_step_idx, end_step_idx = task_step_intervals[t_idx]
        task_col = f"task_{t_idx}_acc"

        task_trajectory = (
            df.iloc[start_step_idx : end_step_idx + 1][task_col].dropna().values
        )
        steps = (
            df.iloc[start_step_idx : end_step_idx + 1]["step"]
            .iloc[: len(task_trajectory)]
            .values
        )

        if len(task_trajectory) > 1:
            auc_val = np.trapezoid(y=task_trajectory, x=steps) / (steps[-1] - steps[0])
        elif len(task_trajectory) == 1:
            auc_val = task_trajectory[0]
        else:
            auc_val = 0.0
        current_task_auc.append(auc_val)

        # 3. Total Basis Rank at task wrap-up
        if "total_basis_rank" in df.columns:
            total_rank_val = df.iloc[boundary_idx]["total_basis_rank"]
        else:
            rank_cols = [c for c in df.columns if c.startswith("basis_rank_")]
            total_rank_val = (
                df.iloc[boundary_idx][rank_cols].sum() if len(rank_cols) > 0 else np.nan
            )
        total_basis_rank.append(total_rank_val)

        # 4. Layer 1 Basis Rank at task wrap-up
        if layer_1_col and layer_1_col in df.columns:
            layer_1_val = df.iloc[boundary_idx][layer_1_col]
        else:
            layer_1_val = np.nan
        layer_1_basis_rank.append(layer_1_val)

    return {
        "avg_acc": avg_acc_at_wrapup,
        "auc": current_task_auc,
        "total_rank": total_basis_rank,
        "layer1_rank": layer_1_basis_rank,
    }


# --- 3. PROCESS BOTH RUNS ---
print(f"Extracting Gaussian baseline metrics from: {GAUSSIAN_CSV}")
gauss_data = extract_run_metrics(GAUSSIAN_CSV, NUM_TASKS)

print(f"Extracting Heavy-Tailed metrics from: {HEAVY_TAIL_CSV}")
ht_data = extract_run_metrics(HEAVY_TAIL_CSV, NUM_TASKS)


# --- 4. 2x2 COMPARATIVE VISUALIZATION ---
fig, axs = plt.subplots(2, 2, figsize=(15, 11), dpi=100)
tasks_axis = np.arange(1, NUM_TASKS + 1)

# Color and style definitions
gauss_color, ht_color = "crimson", "dodgerblue"
gauss_marker, ht_marker = "o", "s"

# [TOP LEFT] Global Cumulative Average Accuracy
axs[0, 0].plot(
    tasks_axis,
    gauss_data["avg_acc"],
    color=gauss_color,
    marker=gauss_marker,
    linewidth=2,
    label="Gaussian",
)
axs[0, 0].plot(
    tasks_axis,
    ht_data["avg_acc"],
    color=ht_color,
    marker=ht_marker,
    linewidth=2,
    label="Heavy-Tailed",
)
axs[0, 0].set_title("1. Global Average Test Accuracy", fontsize=11, weight="bold")
axs[0, 0].set_xlabel("Task Index")
axs[0, 0].set_ylabel("Mean Accuracy across Learned Tasks")
axs[0, 0].set_xticks(tasks_axis)
axs[0, 0].grid(True, linestyle=":", alpha=0.5)
axs[0, 0].legend()

# [TOP RIGHT] Task Plasticity AUC
axs[0, 1].plot(
    tasks_axis,
    gauss_data["auc"],
    color=gauss_color,
    marker=gauss_marker,
    linewidth=2,
    label="Gaussian",
)
axs[0, 1].plot(
    tasks_axis,
    ht_data["auc"],
    color=ht_color,
    marker=ht_marker,
    linewidth=2,
    label="Heavy-Tailed",
)
axs[0, 1].set_title(
    "2. Current Task Learning Curve AUC (Plasticity)", fontsize=11, weight="bold"
)
axs[0, 1].set_xlabel("Task Index")
axs[0, 1].set_ylabel("Normalized Training AUC")
axs[0, 1].set_xticks(tasks_axis)
axs[0, 1].grid(True, linestyle=":", alpha=0.5)
axs[0, 1].legend()

# [BOTTOM LEFT] Total Basis Rank
axs[1, 0].plot(
    tasks_axis,
    gauss_data["total_rank"],
    color=gauss_color,
    marker=gauss_marker,
    linewidth=2,
    label="Gaussian",
)
axs[1, 0].plot(
    tasks_axis,
    ht_data["total_rank"],
    color=ht_color,
    marker=ht_marker,
    linewidth=2,
    label="Heavy-Tailed",
)
axs[1, 0].set_title("3. Cumulative Total Basis Rank", fontsize=11, weight="bold")
axs[1, 0].set_xlabel("Task Index")
axs[1, 0].set_ylabel("Total Reserved Basis Vectors")
axs[1, 0].set_xticks(tasks_axis)
axs[1, 0].grid(True, linestyle=":", alpha=0.5)
axs[1, 0].legend()

# [BOTTOM RIGHT] Layer 1 Basis Rank
axs[1, 1].plot(
    tasks_axis,
    gauss_data["layer1_rank"],
    color=gauss_color,
    marker=gauss_marker,
    linewidth=2,
    label="Gaussian",
)
axs[1, 1].plot(
    tasks_axis,
    ht_data["layer1_rank"],
    color=ht_color,
    marker=ht_marker,
    linewidth=2,
    label="Heavy-Tailed",
)
axs[1, 1].set_title("4. Layer 1 Basis Rank", fontsize=11, weight="bold")
axs[1, 1].set_xlabel("Task Index")
axs[1, 1].set_ylabel("Layer 1 Reserved Basis Vectors")
axs[1, 1].set_xticks(tasks_axis)
axs[1, 1].grid(True, linestyle=":", alpha=0.5)
axs[1, 1].legend()

plt.tight_layout()
plt.savefig("gpm_gaussian_vs_heavytail_comparison.pdf", bbox_inches="tight")
plt.show()

In [ ]:
from pathlib import Path

import torch
from torch import nn
from torchvision import datasets, transforms

# --- 1. CONFIGURATION & PATHS ---
# Update paths to point to your respective snapshot files
PATH_GAUSSIAN = Path("./checkpoints/snapshots/snapshot_A2.0_T20_s0.pt")
PATH_HEAVY_TAILED = Path("./checkpoints/snapshots/snapshot_A1.2_T20_s0.pt")

NUM_TASKS = 20
BATCH_SIZE = 1024  # Size of the evaluation batch per task

# Architecture hyper-parameters
HIDDEN_SIZE = 784
DEPTH = 9
ACTIVATION_NAME = "tanh"
BIAS = False
SEED = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# --- 2. EXTRACTION HELPER FUNCTION ---
def extract_model_and_preactivations(
    snapshot_path: Path,
    test_imgs_raw: torch.Tensor,
    num_tasks: int = 20,
    batch_size: int = 1024,
    device: torch.device = DEVICE,
):
    """Loads a snapshot checkpoint, reconstructs the model, and extracts

    pre-activations across all tasks.
    """
    if not snapshot_path.exists():
        raise FileNotFoundError(f"Snapshot not found at: {snapshot_path}")

    print(f"\n--- Processing Snapshot: {snapshot_path.name} ---")
    snapshot = torch.load(snapshot_path, map_location=device)
    metadata = snapshot.get("metadata", {})
    seed = metadata.get("seed", 0)

    print(
        f"Metadata -> Task: {metadata.get('task')}, Epoch: {metadata.get('epoch')}, Seed: {seed}"
    )

    # Reconstruct architecture and load weights
    model = GeneralMLP(
        input_size=784,
        hidden_size=HIDDEN_SIZE,
        num_classes=10,
        depth=DEPTH,
        activation_name=ACTIVATION_NAME,
        bias=BIAS,
    ).to(device)

    model.load_state_dict(snapshot["state_dict"])
    model.eval()

    # Regenerate task permutations matching the snapshot's seed
    set_seed(SEED)
    task_permutations = generate_permutations(num_tasks=num_tasks, seed=SEED)

    # Extract pre-activations per layer per task
    task_pre_acts = []
    with torch.no_grad():
        for t_idx in range(num_tasks):
            perm = task_permutations[t_idx]
            task_batch = test_imgs_raw[:batch_size, perm]

            pre_acts = model.get_pre_activations(task_batch)

            if isinstance(pre_acts, dict):
                layer_keys = [k for k in pre_acts.keys() if k != "classifier"]
                layer_list = [pre_acts[k] for k in layer_keys]
                if "classifier" in pre_acts:
                    layer_list.append(pre_acts["classifier"])
            else:
                layer_list = pre_acts

            task_pre_acts.append(layer_list)

    print(f"Extraction complete for {snapshot_path.name}.")

    return {
        "model": model,
        "task_pre_acts": task_pre_acts,
        "metadata": metadata,
        "seed": seed,
    }


# --- 3. MAIN EXECUTION PIPELINE ---

# A. Prepare Data (Loaded once for both runs)
mnist_test = datasets.MNIST(
    "../data", train=False, download=True, transform=transforms.ToTensor()
)
test_imgs_raw, _ = get_gpu_data(mnist_test)  # Unpermuted GPU images [N, 784]

# B. Process Both Snapshots
run_gaussian = extract_model_and_preactivations(
    snapshot_path=PATH_GAUSSIAN,
    test_imgs_raw=test_imgs_raw,
    num_tasks=NUM_TASKS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

run_ht = extract_model_and_preactivations(
    snapshot_path=PATH_HEAVY_TAILED,
    test_imgs_raw=test_imgs_raw,
    num_tasks=NUM_TASKS,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

# C. Consolidated Output Dictionary ready for comparative plotting
runs_data = {"gaussian": run_gaussian, "heavy_tailed": run_ht}

print("\nExtraction successfully completed for both Gaussian and Heavy-Tailed runs!")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

# --- 1. CONFIGURATION FOR 2x2 COMPARISON ---

# Select task index to analyze (0-indexed, e.g., 0 for Task 1)
TARGET_TASK_IDX = 0

# Select two layers to compare (0-indexed, e.g., 0 for Layer 1, 4 for Layer 5)
LAYER_IDX_A = 0  # Left Column (e.g., Layer 1)
LAYER_IDX_B = 4  # Right Column (e.g., Layer 5)

ACTIVATION_TYPE = "tanh"  # Options: 'tanh', 'relu'


# --- 2. HELPER FUNCTIONS ---


def get_linear_weights(model):
    """Extracts weight matrices from all linear layers in the model."""
    linear_weights = []
    for module in model.modules():
        if isinstance(module, nn.Linear):
            linear_weights.append(module.weight.detach().cpu().numpy())
    return linear_weights


def compute_activation_derivative(pre_acts, act_type="tanh"):
    """Computes element-wise derivative of activation function given pre-activations."""
    if isinstance(pre_acts, torch.Tensor):
        pre_acts = pre_acts.detach().cpu().numpy()

    if act_type.lower() == "tanh":
        # d/dz tanh(z) = 1 - tanh^2(z)
        return 1.0 - np.tanh(pre_acts) ** 2
    elif act_type.lower() == "relu":
        # d/dz relu(z) = H(z) (Heaviside step)
        return (pre_acts > 0).astype(np.float32)
    else:
        raise ValueError(f"Unsupported activation function: {act_type}")


def compute_layer_jacobian_eigenvalues(
    model, task_pre_acts, task_idx, layer_idx, act_type="tanh"
):
    """Computes the effective layerwise Jacobian J = D @ W and returns its complex eigenvalues.

    J_l = diag(mean_batch(sigma'(z_l))) @ W_l
    """
    weights = get_linear_weights(model)
    W_l = weights[layer_idx]  # Shape: [out_features, in_features]

    # Retrieve pre-activations for chosen task and layer: shape [BATCH_SIZE, hidden_size]
    z_l = task_pre_acts[task_idx][layer_idx]

    # 1. Compute derivative sigma'(z_l)
    sigma_prime = compute_activation_derivative(z_l, act_type=act_type)

    # 2. Average derivative gate over batch dimension -> shape: [hidden_size]
    d_l = np.mean(sigma_prime, axis=0)

    # 3. Construct effective Jacobian operator: J_l = diag(d_l) @ W_l
    J_l = np.diag(d_l) @ W_l

    # 4. Compute full complex eigenvalue spectrum
    eigenvalues = np.linalg.eigvals(J_l)

    return eigenvalues


# --- 3. COMPUTATION & SPECTRUM EXTRACTION FOR BOTH RUNS ---

print(f"Computing 2x2 Jacobian spectra for Task {TARGET_TASK_IDX + 1}...")

# Row 1: Gaussian Initialisation
eigs_gauss_A = compute_layer_jacobian_eigenvalues(
    runs_data["gaussian"]["model"],
    runs_data["gaussian"]["task_pre_acts"],
    TARGET_TASK_IDX,
    LAYER_IDX_A,
    act_type=ACTIVATION_TYPE,
)
eigs_gauss_B = compute_layer_jacobian_eigenvalues(
    runs_data["gaussian"]["model"],
    runs_data["gaussian"]["task_pre_acts"],
    TARGET_TASK_IDX,
    LAYER_IDX_B,
    act_type=ACTIVATION_TYPE,
)

# Row 2: Heavy-Tailed Initialisation
eigs_ht_A = compute_layer_jacobian_eigenvalues(
    runs_data["heavy_tailed"]["model"],
    runs_data["heavy_tailed"]["task_pre_acts"],
    TARGET_TASK_IDX,
    LAYER_IDX_A,
    act_type=ACTIVATION_TYPE,
)
eigs_ht_B = compute_layer_jacobian_eigenvalues(
    runs_data["heavy_tailed"]["model"],
    runs_data["heavy_tailed"]["task_pre_acts"],
    TARGET_TASK_IDX,
    LAYER_IDX_B,
    act_type=ACTIVATION_TYPE,
)

# Global axis limit synchronization across ALL 4 panels
all_eigs = [eigs_gauss_A, eigs_gauss_B, eigs_ht_A, eigs_ht_B]
max_global_radius = max(np.max(np.abs(e)) for e in all_eigs)
axis_limit = max(max_global_radius, 1.2) * 1.1  # Ensures unit circle fits cleanly


# --- 4. 2x2 GRID COMPLEX PLANE PLOTTING ---

fig, axes = plt.subplots(2, 2, figsize=(14, 13))

# Unit circle parametrization
theta = np.linspace(0, 2 * np.pi, 300)
unit_circle_x = np.cos(theta)
unit_circle_y = np.sin(theta)

# Define 2x2 grid panel specifications
grid_specs = [
    # (Row, Col, Eigenvalues, Layer Index, Label, Accent Color)
    (
        0,
        0,
        eigs_gauss_A,
        LAYER_IDX_A,
        f"Gaussian Init — Layer {LAYER_IDX_A + 1}",
        "teal",
    ),
    (
        0,
        1,
        eigs_gauss_B,
        LAYER_IDX_B,
        f"Gaussian Init — Layer {LAYER_IDX_B + 1}",
        "darkcyan",
    ),
    (
        1,
        0,
        eigs_ht_A,
        LAYER_IDX_A,
        f"Heavy-Tailed Init — Layer {LAYER_IDX_A + 1}",
        "indigo",
    ),
    (
        1,
        1,
        eigs_ht_B,
        LAYER_IDX_B,
        f"Heavy-Tailed Init — Layer {LAYER_IDX_B + 1}",
        "crimson",
    ),
]

for row, col, eigs, l_idx, title_prefix, color in grid_specs:
    ax = axes[row, col]

    # 1. Reference grid & origin lines
    ax.axhline(0, color="gray", linestyle=":", alpha=0.6, linewidth=1.0)
    ax.axvline(0, color="gray", linestyle=":", alpha=0.6, linewidth=1.0)

    # 2. Unit circle boundary (|lambda| = 1)
    ax.plot(
        unit_circle_x,
        unit_circle_y,
        color="black",
        linestyle="--",
        linewidth=1.8,
        label=r"Unit Boundary ($|\lambda| = 1$)",
        zorder=2,
    )

    # 3. Scatter plot complex eigenvalues
    ax.scatter(
        eigs.real,
        eigs.imag,
        color=color,
        alpha=0.65,
        s=26,
        edgecolors="none",
        label=f"Eigenvalues (N={len(eigs)})",
        zorder=3,
    )

    # 4. Diagnostics: Spectral radius and zero mass density
    zero_count = np.sum(np.abs(eigs) < 0.05)
    zero_ratio = (zero_count / len(eigs)) * 100
    max_r = np.max(np.abs(eigs))

    # Formatting & Limits
    ax.set_aspect("equal", "box")
    ax.set_xlim(-axis_limit, axis_limit)
    ax.set_ylim(-axis_limit, axis_limit)

    ax.set_xlabel(r"Real Part $\operatorname{Re}(\lambda)$", fontweight="bold")
    ax.set_ylabel(r"Imaginary Part $\operatorname{Im}(\lambda)$", fontweight="bold")

    ax.set_title(
        f"{title_prefix}",
        # f"Max Radius $\\rho(J) = {max_r:.2f}$ | Zero-Mass ($|\\lambda|<0.05$): {zero_ratio:.1f}%",
        fontsize=11,
        fontweight="bold",
        pad=10,
    )
    ax.grid(True, linestyle=":", alpha=0.4)
    ax.legend(loc="upper right", frameon=True, fontsize=8.5)

plt.suptitle(
    f"Layerwise Non-Hermitian Jacobian Spectra Comparison (Task {TARGET_TASK_IDX + 1})\n"
    f"Gaussian (Row 1) vs. Heavy-Tailed (Row 2) Across Layer Depth",
    fontsize=14,
    fontweight="bold",
    y=0.99,
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# --- 1. CONFIGURATION ---

TARGET_TASK_IDX = 2  # Task index to analyze (0-indexed)
ACTIVATION_TYPE = "tanh"  # Options: 'tanh', 'relu'


# --- 2. HELPER FUNCTIONS ---


def get_linear_weights(model):
    """Extracts weight matrices from all linear layers in the model."""
    linear_weights = []
    for module in model.modules():
        if isinstance(module, nn.Linear):
            linear_weights.append(module.weight.detach().cpu().numpy())
    return linear_weights


def compute_activation_derivative(pre_acts, act_type="tanh"):
    """Computes element-wise derivative of activation function given pre-activations."""
    if isinstance(pre_acts, torch.Tensor):
        pre_acts = pre_acts.detach().cpu().numpy()

    if act_type.lower() == "tanh":
        # d/dz tanh(z) = 1 - tanh^2(z)
        return 1.0 - np.tanh(pre_acts) ** 2
    elif act_type.lower() == "relu":
        # d/dz relu(z) = H(z) (Heaviside step)
        return (pre_acts > 0).astype(np.float32)
    else:
        raise ValueError(f"Unsupported activation function: {act_type}")


def compute_layer_jacobian_eigenvalues(
    model, task_pre_acts, task_idx, layer_idx, act_type="tanh"
):
    """Computes the effective layerwise Jacobian J = D @ W and returns its complex eigenvalues."""
    weights = get_linear_weights(model)
    W_l = weights[layer_idx]  # Shape: [out_features, in_features]

    # Retrieve pre-activations for chosen task and layer
    z_l = task_pre_acts[task_idx][layer_idx]

    # 1. Compute derivative sigma'(z_l)
    sigma_prime = compute_activation_derivative(z_l, act_type=act_type)

    # 2. Average derivative gate over batch dimension
    d_l = np.mean(sigma_prime, axis=0)

    # 3. Construct effective Jacobian operator: J_l = diag(d_l) @ W_l
    J_l = np.diag(d_l) @ W_l

    # 4. Compute full complex eigenvalue spectrum
    eigenvalues = np.linalg.eigvals(J_l)

    return eigenvalues


# --- 3. CORE METRICS COMPUTATION FUNCTION ---


def compute_jacobian_metrics(eigenvalues):
    """Computes the 5 core quantitative metrics for a set of complex eigenvalues:

    1. Max Spectral Radius (r_max)
    2. Fraction of Active Outliers (% |lambda| > 1)
    3. Effective Rank / Spectral Entropy (k_eff)
    4. Gini Coefficient of Spectral Energy (G)
    5. Median to Mean Magnitude Ratio (Med/Mean)
    """
    mags = np.abs(eigenvalues)
    N = len(mags)

    # 1. Max Spectral Radius
    r_max = np.max(mags)

    # 2. Active Outliers Percentage (|lambda| > 1.0)
    pct_outliers = (np.sum(mags > 1.0) / N) * 100.0

    # 3. Effective Rank (k_eff) via Spectral Entropy
    # Avoid log(0) by adding eps to zero magnitudes
    mags_clean = np.where(mags == 0, 1e-12, mags)
    p = mags_clean / np.sum(mags_clean)
    spectral_entropy = -np.sum(p * np.log(p))
    k_eff = np.exp(spectral_entropy)

    # 4. Gini Coefficient (G)
    sorted_mags = np.sort(mags)
    index = np.arange(1, N + 1)
    gini = (2.0 * np.sum(index * sorted_mags)) / (N * np.sum(sorted_mags)) - (
        N + 1.0
    ) / N

    # 5. Median to Mean Ratio
    mean_mag = np.mean(mags)
    median_mag = np.median(mags)
    med_mean_ratio = median_mag / mean_mag if mean_mag > 0 else 0.0

    return {
        "r_max": r_max,
        "pct_outliers": pct_outliers,
        "k_eff": k_eff,
        "gini": gini,
        "med_mean_ratio": med_mean_ratio,
    }


# --- 4. EXECUTION ACROSS ALL LAYERS FOR HEAVY-TAILED MODEL ---

ht_model = runs_data["heavy_tailed"]["model"]
ht_pre_acts = runs_data["heavy_tailed"]["task_pre_acts"]
num_layers = len(get_linear_weights(ht_model))

results_list = []

print(
    f"Calculating 5 Core Metrics across ALL {num_layers} Layers for Heavy-Tailed Model (Task {TARGET_TASK_IDX + 1})...\n"
)

for layer_idx in range(num_layers - 1):  # Exclude final output layer
    # Compute complex eigenvalues for layer
    eigs = compute_layer_jacobian_eigenvalues(
        ht_model,
        ht_pre_acts,
        TARGET_TASK_IDX,
        layer_idx,
        act_type=ACTIVATION_TYPE,
    )

    # Extract metrics
    metrics = compute_jacobian_metrics(eigs)

    results_list.append(
        {
            "Layer": f"Layer {layer_idx + 1}",
            "N (Dim)": len(eigs),
            "Max Radius (r_max)": f"{metrics['r_max']:.3f}",
            "Active Outliers (|λ|>1)": f"{metrics['pct_outliers']:.2f}%",
            "Effective Rank (k_eff)": f"{metrics['k_eff']:.1f}",
            "Gini Index (G)": f"{metrics['gini']:.3f}",
            "Med / Mean Ratio": f"{metrics['med_mean_ratio']:.3f}",
        }
    )

# Format into DataFrame
df_metrics = pd.DataFrame(results_list)

# Display Formatted Table
print("=" * 85)
print(f"HEAVY-TAILED JACOBIAN SPECTRAL METRICS (TASK {TARGET_TASK_IDX + 1})")
print("=" * 85)
print(df_metrics.to_string(index=False))
print("=" * 85)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch

# --- 1. CONFIGURATION FOR SINGLE LAYER & TASK ---
LAYER_IDX = 4  # e.g., 5th layer (index 4 in pre_acts list, 'features_8_weight')
TASK_IDX = 1  # Evaluate Task 1 against Task 0 basis
MAX_MODES = 50  # Top Jacobian eigenmodes to evaluate


def compute_jacobian_basis_alignment(
    run_dict,
    snapshot_path,
    task_idx=1,
    layer_query="features_8_weight",
    max_modes=50,
    verbose=True,
):
    """Computes alignment between layerwise Jacobian eigenmodes and the GPM basis

    with explicit verification of layer names, shapes, and task indices.
    """
    snapshot_path = Path(snapshot_path)
    snap = torch.load(snapshot_path, map_location="cpu", weights_only=False)

    # -------------------------------------------------------------------------
    # 1. Resolve Explicit Parameter & Basis Keys
    # -------------------------------------------------------------------------
    model = run_dict["model"]
    named_weights = {
        name: p
        for name, p in model.named_parameters()
        if "weight" in name and "classifier" not in name
    }
    classifier_weights = {
        name: p for name, p in model.named_parameters() if "classifier" in name
    }

    basis_dict = snap["current_basis"]

    # Match layer query to model weight key
    query_str = str(layer_query).replace(".", "_")
    matched_param_name = None
    matched_basis_key = None

    # Check linear/conv layers
    for p_name in list(named_weights.keys()) + list(classifier_weights.keys()):
        clean_p_name = p_name.replace(".", "_")
        if query_str in clean_p_name or str(layer_query) == clean_p_name:
            matched_param_name = p_name
            break

    if matched_param_name is None:
        # Fallback by numeric index if an int or string digit was passed
        digits = re.findall(r"\d+", query_str)
        if digits:
            idx = int(digits[0])
            all_weight_keys = list(named_weights.keys())
            matched_param_name = all_weight_keys[min(idx, len(all_weight_keys) - 1)]
        else:
            raise KeyError(f"Could not resolve layer query '{layer_query}'. Available: {list(named_weights.keys())}")

    # Match corresponding basis key
    for b_key in basis_dict.keys():
        if matched_param_name.replace(".", "_") == b_key.replace(".", "_"):
            matched_basis_key = b_key
            break

    if matched_basis_key is None:
        raise KeyError(
            f"Param '{matched_param_name}' not found in snapshot basis keys: {list(basis_dict.keys())}"
        )

    # -------------------------------------------------------------------------
    # 2. Extract Matrices and Pre-activations
    # -------------------------------------------------------------------------
    # Determine layer index integer for pre-activations list
    all_weight_names = list(named_weights.keys())
    if matched_param_name in all_weight_names:
        layer_ordinal = all_weight_names.index(matched_param_name)
    else:
        layer_ordinal = -1  # Classifier

    W_tensor = model.state_dict()[matched_param_name]
    W = W_tensor.detach().cpu().numpy().astype(np.float64)

    # Basis M (Shape: [d_in, K])
    M = basis_dict[matched_basis_key].detach().cpu().numpy().astype(np.float64)
    if M.ndim == 1:
        M = M[:, np.newaxis]

    # Pre-activations h (Shape: [Batch, d_out])
    pre_act = run_dict["task_pre_acts"][task_idx][layer_ordinal].detach().cpu().float()
    d_diag = (1.0 - torch.tanh(pre_act) ** 2).mean(dim=0).numpy().astype(np.float64)

    # -------------------------------------------------------------------------
    # 3. Construct Layerwise Jacobian J = diag(d) @ W
    # -------------------------------------------------------------------------
    J = np.diag(d_diag) @ W

    # -------------------------------------------------------------------------
    # 4. Eigendecomposition and Alignment Projection
    # -------------------------------------------------------------------------
    eigvals, eigvecs = np.linalg.eig(J)

    sort_idx = np.argsort(np.abs(eigvals))[::-1]
    eigvals_sorted = eigvals[sort_idx]
    eigvecs_sorted = eigvecs[:, sort_idx]

    num_eval = min(max_modes, len(eigvals_sorted))
    a_k = np.zeros(num_eval)
    moduli = np.abs(eigvals_sorted[:num_eval])

    for k in range(num_eval):
        vk = eigvecs_sorted[:, k]
        vk = vk / np.linalg.norm(vk)
        # Complex Hermitian inner product
        proj_coords = M.T @ vk
        a_k[k] = np.real(np.sum(np.conj(proj_coords) * proj_coords))

    # -------------------------------------------------------------------------
    # 5. Diagnostic Verification Logging
    # -------------------------------------------------------------------------
    if verbose:
        snap_meta = snap.get("metadata", {})
        snap_task = snap_meta.get("task_idx", snap_meta.get("task", "Unknown"))
        print("=" * 70)
        print("JACOBIAN-TO-BASIS ALIGNMENT DIAGNOSTICS")
        print("=" * 70)
        print(f"Target Layer Query    : {layer_query}")
        print(f"Model Parameter Key   : {matched_param_name}")
        print(f"Snapshot Basis Key    : {matched_basis_key}")
        print(f"Snapshot Task Index   : Task {snap_task} (Basis Rank K = {M.shape[1]})")
        print(f"Evaluated Pre-Act Task: Task {task_idx}")
        print("-" * 70)
        print(f"Weight Shape (W)      : {W.shape} (Out: {W.shape[0]}, In: {W.shape[1]})")
        print(f"Derivative Vector (D) : {d_diag.shape} (Mean slope: {d_diag.mean():.4f})")
        print(f"Jacobian Matrix (J)   : {J.shape}")
        print(f"Basis Matrix (M)      : {M.shape} (Ambient: {M.shape[0]}, Subspace dim: {M.shape[1]})")
        print("-" * 70)
        print(f"Leading Eigenvalue |λ₁|: {moduli[0]:.4f} (Max transmission gain)")
        print(f"Mode 1 Overlap (a₁)   : {a_k[0]:.4f} ({a_k[0]*100:.2f}% into basis)")
        print(f"Mean Top-5 Overlap    : {a_k[:5].mean():.4f} ({a_k[:5].mean()*100:.2f}%)")
        print("=" * 70)

    return {
        "a_k": np.clip(a_k, 0.0, 1.0),
        "moduli": moduli,
        "eigvals": eigvals_sorted,
        "basis_dim": M.shape[1],
        "param_name": matched_param_name,
        "basis_key": matched_basis_key,
    }

# --- 2. EXECUTION ---
# 1. Load the initial task snapshot (Task 0 basis, M_0)
PATH_HT_T0 = Path("./checkpoints/snapshots/snapshot_A1.2_T01_s0.pt")
PATH_G_T0 = Path("./checkpoints/snapshots/snapshot_A2.0_T01_s0.pt")

# 2. Extract Task 0 model and Task 0 pre-activations
run_ht_t0 = extract_model_and_preactivations(
    snapshot_path=PATH_HT_T0,
    test_imgs_raw=test_imgs_raw,
    num_tasks=1,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

run_g_t0 = extract_model_and_preactivations(
    snapshot_path=PATH_G_T0,
    test_imgs_raw=test_imgs_raw,
    num_tasks=1,
    batch_size=BATCH_SIZE,
    device=DEVICE,
)

# 3. Compute alignment using synchronized Task 0 states
res_ht = compute_jacobian_basis_alignment(
    run_dict=run_ht_t0,
    snapshot_path=PATH_HT_T0,
    task_idx=0,
    layer_query="features.8.weight",
    max_modes=40,
    verbose=True,
)

res_g = compute_jacobian_basis_alignment(
    run_dict=run_g_t0,
    snapshot_path=PATH_G_T0,
    task_idx=0,
    layer_query="features.8.weight",
    max_modes=40,
    verbose=True,
)

# --- 3. TWO-PANEL DIAGNOSTIC PLOT ---
modes = np.arange(1, MAX_MODES + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2), dpi=150)

# Left: Jacobian Eigenmode Alignment with Basis
ax1.plot(
    modes,
    res_ht["a_k"],
    marker="s",
    lw=2.2,
    color="#1f77b4",
    label=rf"Heavy-Tailed ($\alpha=1.2$, $K={res_ht['basis_dim']}$)",
)
ax1.plot(
    modes,
    res_g["a_k"],
    marker="o",
    lw=2.2,
    color="#d62728",
    label=rf"Gaussian ($\alpha=2.0$, $K={res_g['basis_dim']}$)",
)
ax1.set_title(
    f"Jacobian Eigenmode Alignment: $a_k = \\|\\mathbf{{M}}^T \\mathbf{{v}}_k\\|^2$\nLayer: {res_ht['basis_key']} (Task {TASK_IDX})",
    fontsize=10.5,
    fontweight="bold",
)
ax1.set_xlabel("Jacobian Mode Index ($k$, sorted by $|\\lambda_k|$)", fontsize=10)
ax1.set_ylabel(r"Projection onto GPM Basis ($a_k$)", fontsize=10)
ax1.set_ylim(-0.02, 1.05)
ax1.grid(True, linestyle="--", alpha=0.4)
ax1.legend(frameon=True, fontsize=8.5)

# Right: Transmission Gain |lambda_k|
ax2.plot(
    modes,
    res_ht["moduli"],
    marker="s",
    lw=2.2,
    color="#1f77b4",
    label=r"Heavy-Tailed ($\alpha=1.2$)",
)
ax2.plot(
    modes,
    res_g["moduli"],
    marker="o",
    lw=2.2,
    color="#d62728",
    label=r"Gaussian ($\alpha=2.0$)",
)
ax2.axhline(1.0, color="gray", linestyle="--", lw=1.2, label=r"Critical Boundary ($|\lambda|=1$)")
ax2.set_title(
    f"Jacobian Transmission Gain ($|\\lambda_k|$)\nLayer: {res_ht['basis_key']}",
    fontsize=10.5,
    fontweight="bold",
)
ax2.set_xlabel("Jacobian Mode Index ($k$)", fontsize=10)
ax2.set_ylabel(r"Eigenvalue Modulus $|\lambda_k|$", fontsize=10)
ax2.grid(True, linestyle="--", alpha=0.4)
ax2.legend(frameon=True, fontsize=8.5)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import torch
import torchvision
import torchvision.transforms as transforms
from scipy.special import digamma
from sklearn.neighbors import NearestNeighbors

# --- 1. CORE DIMENSIONALITY ESTIMATORS ---


def compute_participation_ratio(X):
    """Computes Participation Ratio (PR) from centered data/activations.

    PR = (sum(lambda_i))^2 / sum(lambda_i^2) = (sum(s_i^2))^2 / sum(s_i^4)
    Measures the continuous linear spectral spread without arbitrary variance thresholds.
    """
    # Center data across batch dimension
    X_centered = X - np.mean(X, axis=0, keepdims=True)

    # Compute singular values (S)
    _, S, _ = np.linalg.svd(X_centered, full_matrices=False)

    S_sq = S**2
    sum_S_sq = np.sum(S_sq)
    sum_S_quad = np.sum(S_sq**2)

    if sum_S_quad < 1e-12:
        return 0.0

    return (sum_S_sq**2) / sum_S_quad


def compute_gride_id(X, k1=10, k2=20):
    """Computes non-linear Intrinsic Dimension using GRIDE (Generalized Ratio Estimator).

    MLE formula: d = [digamma(k2) - digamma(k1)] / mean(log(r_k2 / r_k1))
    Captures manifold topology while filtering out microscopic noise.
    """
    nbrs = NearestNeighbors(n_neighbors=k2 + 1, algorithm="auto").fit(X)
    distances, _ = nbrs.kneighbors(X)

    r1 = distances[:, k1]
    r2 = distances[:, k2]

    valid_mask = (r1 > 1e-12) & (r2 > r1)
    if not np.any(valid_mask):
        return 0.0

    log_ratios = np.log(r2[valid_mask] / r1[valid_mask])
    mean_log_ratio = np.mean(log_ratios)

    digamma_diff = digamma(k2) - digamma(k1)
    return digamma_diff / mean_log_ratio


# --- 2. RAW MNIST BASELINE COMPUTATION ---

print("Fetching raw MNIST dataset for ground-truth reference baseline...")
transform = transforms.Compose([transforms.ToTensor()])

# Download and load MNIST test set
mnist_dataset = torchvision.datasets.MNIST(
    root="../data", train=False, download=True, transform=transform
)

# Extract a 5,000-sample evaluation batch
eval_loader = torch.utils.data.DataLoader(mnist_dataset, batch_size=5000, shuffle=False)
raw_images, _ = next(iter(eval_loader))

# Flatten images from [5000, 1, 28, 28] to 2D matrix [5000, 784]
raw_mnist_samples = raw_images.view(raw_images.size(0), -1).numpy()

# Compute raw dataset baseline metrics
raw_mnist_pr = compute_participation_ratio(raw_mnist_samples)
raw_mnist_gride = compute_gride_id(raw_mnist_samples, k1=10, k2=20)

print("=" * 70)
print("GROUND-TRUTH RAW MNIST BASELINE")
print(f"  • Ambient Space Dimension (N):     784")
print(f"  • Raw Continuous Spectral PR:      {raw_mnist_pr:.1f} (Linear Spread)")
print(
    f"  • Raw Non-Linear GRIDE ID:          {raw_mnist_gride:.2f} (Manifold Topological Dim)"
)
print("=" * 70 + "\n")


# --- 3. LAYERWISE MODEL EVALUATION ROUTINE ---

TARGET_TASK_IDX = 0  # Select task index (0-indexed)
ACTIVATION_FN = torch.tanh

# Retrieve pre-activations for both models
gauss_pre_task = runs_data["gaussian"]["task_pre_acts"][TARGET_TASK_IDX]
ht_pre_task = runs_data["heavy_tailed"]["task_pre_acts"][TARGET_TASK_IDX]

num_layers = len(ht_pre_task)
metrics_log = []

print(
    f"Calculating PR & GRIDE ID for Task {TARGET_TASK_IDX + 1} across {num_layers} layers...\n"
)

for l_idx in range(num_layers):
    g_pre = gauss_pre_task[l_idx]
    ht_pre = ht_pre_task[l_idx]

    # Convert to post-activations: [N_samples, hidden_dim]
    if isinstance(g_pre, torch.Tensor):
        g_post = ACTIVATION_FN(g_pre).detach().cpu().numpy()
    else:
        g_post = ACTIVATION_FN(torch.tensor(g_pre)).numpy()

    if isinstance(ht_pre, torch.Tensor):
        ht_post = ACTIVATION_FN(ht_pre).detach().cpu().numpy()
    else:
        ht_post = ACTIVATION_FN(torch.tensor(ht_pre)).numpy()

    # 1. Compute Linear Continuous Dimensionality (Participation Ratio)
    g_pr = compute_participation_ratio(g_post)
    ht_pr = compute_participation_ratio(ht_post)

    # 2. Compute Non-Linear Topological Dimensionality (GRIDE)
    g_gride = compute_gride_id(g_post, k1=10, k2=20)
    ht_gride = compute_gride_id(ht_post, k1=10, k2=20)

    metrics_log.append(
        {
            "Layer": f"Layer {l_idx + 1}",
            "Ambient (N)": ht_post.shape[1],
            "Gauss PR": f"{g_pr:.1f}",
            "HT PR": f"{ht_pr:.1f}",
            "PR Shift": f"{ht_pr - g_pr:+.1f}",
            "Gauss GRIDE": f"{g_gride:.2f}",
            "HT GRIDE": f"{ht_gride:.2f}",
            "GRIDE Shift": f"{ht_gride - g_gride:+.2f}",
        }
    )

# Format into DataFrame
df_results = pd.DataFrame(metrics_log)

# Display Formatted Comparative Table
print("=" * 105)
print(f"LAYERWISE DIMENSIONALITY PROFILE COMPARISON (TASK {TARGET_TASK_IDX + 1})")
print(f"Reference Baseline -> Raw MNIST GRIDE: {raw_mnist_gride:.2f}")
print("=" * 105)
print(df_results.to_string(index=False))
print("=" * 105)